# Stat 220, Unit 6 Homework: Inference Without a Formula

`delivery_routes.csv` is the 500 routes from the Unit 4 homework. `minutes` is how long
a route took. Five columns describe the route: `stops`, `packages`, `miles`, `downtown`, `rain`.
Three more are recorded and have nothing to do with the outcome: `van_age_years`,
`dispatcher_rating`, `month`.

Run the next two cells first.

In [ ]:
%pip install -q numpy pandas statsmodels scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
PREDICTORS = ["stops", "packages", "miles", "downtown", "rain",
              "van_age_years", "dispatcher_rating", "month"]
routes.head()

**Problem 1.** *A forest on the routes.* Fit one, and find out how well it does.

Part a. Report the held-out $R^2$ and the out-of-bag $R^2$. Say in one sentence what the out-of-bag number is and why it costs nothing to compute.

In [ ]:
X = routes[PREDICTORS]
y = routes["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

forest = RandomForestRegressor(n_estimators=300, random_state=0, oob_score=True,
                               n_jobs=-1).fit(X_train, y_train)
print("held-out R2:", round(forest.score(X_test, y_test), 3))
print("out-of-bag R2:", round(forest.oob_score_, 3))

_Your answer:_



Part b. The two numbers are close. Say what it would mean if the out-of-bag number were much higher than the held-out one.

_Your answer:_



**Problem 2.** *Which columns is it using?* Compare the two importance methods.

Part a. Report both rankings in one table.

In [ ]:
impurity = pd.Series(forest.feature_importances_, index=PREDICTORS)
perm = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=0, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=PREDICTORS)

print(pd.DataFrame({"impurity": impurity.round(4),
                    "permutation": permutation.round(4)}).sort_values("impurity",
                                                                      ascending=False))

_Your answer:_



Part b. Both methods put `month` above `downtown` and `rain`. Two of those three columns are real and one is not. Say which, and explain in two or three sentences how a real predictor can finish below a column that means nothing.

_Your answer:_



**Problem 3.** *Is the ranking an answer?* Fit a regression on the same eight columns.

Part a. Report the coefficient and p-value for `downtown`, `rain` and `month`.

In [ ]:
fit = smf.ols("minutes ~ " + " + ".join(PREDICTORS), data=routes).fit()
for c in ["downtown", "rain", "month"]:
    print(f"{c:<10} coef {fit.params[c]:+7.3f}   p {fit.pvalues[c]:.2g}")

_Your answer:_



Part b. The forest ranked `month` above `rain`. The regression says something different. Explain in three or four sentences what each of the two is measuring, and which one answers the question 'does rain add time to a route'.

_Your answer:_



**Problem 4.** *An interval on an importance score.* A ranking does not say whether a column is
distinguishable from one that means nothing. Resampling does. Resample the training rows only, and
score against the test set you already have.

Part a. Report the point estimate and a 95% bootstrap interval for each column, and say which ones you cannot separate from zero.

In [ ]:
rng = np.random.default_rng(6)

def importance_of(X_fit, y_fit, seed):
    f = RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1).fit(X_fit, y_fit)
    r = permutation_importance(f, X_test, y_test, n_repeats=4, random_state=seed, n_jobs=-1)
    return pd.Series(r.importances_mean, index=PREDICTORS)

base = importance_of(X_train, y_train, 0)
reps = pd.DataFrame([importance_of(X_train.iloc[take], y_train.iloc[take], k)
                     for k, take in enumerate(rng.integers(0, len(X_train),
                                                           (30, len(X_train))))])

for c in PREDICTORS:
    lo, hi = np.percentile(reps[c], [2.5, 97.5])
    flag = "  cannot separate from zero" if lo <= 0 <= hi else ""
    print(f"{c:<18} {base[c]:+.4f}  [{lo:+.4f}, {hi:+.4f}]{flag}")

_Your answer:_



Part b. In problem 3 the regression found `rain` real, with about seven extra minutes and a small p-value. Here the importance of `rain` cannot be separated from zero. Both results are correct. Explain in three or four sentences how that happens, and say which of the two you would report to someone asking whether rain slows deliveries.

_Your answer:_



**Problem 5.** *What can and cannot be said.* No computer needed.

Part a. A colleague fits a forest to customer data and reports that `account_number` is the second most important variable, using the default importance. Write the two or three sentences you would say in response.

_Your answer:_



Part b. Another colleague bootstraps a sample of 40 patients, gets a 95% interval, and says it shows the treatment works in the wider population. The 40 patients were all recruited from one clinic. Explain what the interval does and does not cover.

_Your answer:_



Part c. A third colleague resamples the whole dataset, then splits each resample into training and test rows, and reports bootstrap intervals for permutation importance. Every column comes out important. Say what went wrong.

_Your answer:_

